# Habit Formation

The external habit model of [Campbell and Cochrane (1999)](https://doi.org/10.1086/250059), adapted from an example of EconPDEs.jl. The second part, added in Pollis, computes the equity premium and the Sharpe ratio and sets them beside published tables.

In [ ]:
Pollis.packages("EconPDEs", "Plots")

## From EconPDEs.jl

The code in this part is the package's example, with Greek letters written in Latin (`mu`, `sigma`, `gamma`, `rho`, `kappa`, `lambda`). A line changed in any other way says `# Changed in Pollis:`.

### The model

The state is the log surplus consumption ratio `s = log((C - X) / C)`, how far consumption `C` sits above the habit `X`, which reverts to a mean `sbar`. The unknown is the price-consumption ratio `p(s)`, which solves a linear second-order equation:

```
0 = 1 + (mu + mup(s) + sigmap(s) sigma - r(s) - kappa(s) (sigma + sigmap(s))) p(s)
```

In [ ]:
using EconPDEs, Plots

Base.@kwdef struct CampbellCochraneModel
    mu::Float64 = 0.0189    # mean consumption growth
    sigma::Float64 = 0.015     # volatility of consumption growth
    gamma::Float64 = 2.0       # utility curvature (relative risk aversion)
    rho::Float64 = 0.116     # discount rate
    kappas::Float64 = 0.138    # mean-reversion speed of log surplus consumption ratio
    b::Float64 = 0.0       # sensitivity of riskless rate to surplus ratio (b=0 means constant r)
end

In [ ]:
m = CampbellCochraneModel()

### The grid and the guess

The grid has many points near the upper bound `smax`, and runs far into the left tail, down to `s = -300`. The reach of that tail matters, as the second part shows.

In [ ]:
function initialize_stategrid(m::CampbellCochraneModel; sn = 1000)
    (; mu, sigma, gamma, rho, kappas, b) = m
    Sbar = sigma * sqrt(gamma / (kappas - b / gamma))
    sbar = log(Sbar)
    smax = sbar + 0.5 * (1 - Sbar^2)
    shigh = log.(range(0.0, exp(smax), length = div(sn, 10)))
    slow = range(-300.0, shigh[2], length = sn - div(sn, 10))
    (; s = vcat(slow[1:(end-1)], shigh[2:end]))
end

stategrid = initialize_stategrid(m)

In [ ]:
guess = (; p = ones(length(stategrid[:s])))

### The equation

The first derivative is upwinded on the risk-adjusted drift of `s`, since pricing uses that drift and not the physical one.

In [ ]:
function (m::CampbellCochraneModel)(state::NamedTuple, u::NamedTuple)
    (; mu, sigma, gamma, rho, kappas, b) = m
    (; s) = state
    (; p, ps_up, ps_down, pss) = u
    Sbar = sigma * sqrt(gamma / (kappas - b / gamma))
    sbar = log(Sbar)
    lambda = 1 / Sbar * sqrt(1 - 2 * (s - sbar)) - 1
    mus = -kappas * (s - sbar)
    sigmas = lambda * sigma
    # pricing uses the risk-adjusted drift of surplus consumption, not the physical one
    mus_effective = mus + sigmas * (sigma - gamma * (sigma + sigmas))
    ps = (mus_effective >= 0) ? ps_up : ps_down
    sigmap = ps / p * sigmas
    mup = ps / p * mus + 0.5 * pss / p * sigmas^2
    kappa = gamma * (sigma + sigmas)
    r = rho + gamma * mu - (gamma * kappas - b) / 2 + b * (sbar - s)
    pt = -p * (1 / p + mu + mup + sigmap * sigma - r - kappa * (sigma + sigmap))
    return (; pt), (; kappa, sigmap, r)  # Changed in Pollis: also keeps kappa, sigmap and r
end

### Solving it

In [ ]:
result = pdesolve(m, stategrid, guess)

### The solution

The price-consumption ratio rises with the surplus consumption ratio `S = exp(s)`: as consumption pulls away from the habit, effective risk aversion falls, discount rates drop and valuations climb.

In [ ]:
Sbar = m.sigma * sqrt(m.gamma / (m.kappas - m.b / m.gamma))
sbar = log(Sbar)
ss = stategrid[:s]
mask = ss .>= (sbar - 4)
plot(exp.(ss[mask]), result.solution.p[mask]; xlabel = "surplus consumption ratio S = exp(s)", ylabel = "price-consumption ratio p", legend = false)

## Added in Pollis, not in EconPDEs.jl

Everything from here to the end was written for Pollis. It is not in the package, and the package's author did not write or review it. Each code cell starts with a comment saying so.

The example stops at the price-consumption ratio. The dividend of the claim priced here is consumption itself, which is the **Consumption Claim** column of Campbell and Cochrane's Table 2.

### The equity premium and the Sharpe ratio

With one shock, the claim's expected excess return is `kappa * (sigma + sigmap)`, and its Sharpe ratio is the price of risk `kappa`. Both rise as the surplus falls, the countercyclical pattern the paper is built around.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

premium = result.saved.kappa .* (m.sigma .+ result.saved.sigmap)
plot(plot(exp.(ss[mask]), 100 .* premium[mask]; xlabel = "surplus consumption ratio S", ylabel = "equity premium, %", legend = false),
     plot(exp.(ss[mask]), result.saved.kappa[mask]; xlabel = "surplus consumption ratio S", ylabel = "Sharpe ratio", legend = false);
     layout = (1, 2), size = (820, 340), left_margin = 6Plots.mm, bottom_margin = 6Plots.mm)

### Simulating the statistics of Table 2

The note to Table 2 says the model is simulated at a monthly frequency and the statistics come from time-averaged data at an annual frequency. The function below steps the continuous-time solution monthly for 100,000 years. Its choices for the aggregation are ours: annual returns as the product of monthly returns, annual consumption as the sum over the year, and the price/dividend ratio as the price at the end of the year over that year's consumption.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

using Printf, Random, Statistics

function table2_statistics(m, result, ss; months = 1_200_000, seed = 1)
    (; mu, sigma, gamma, kappas) = m
    p = result.solution.p
    premium = result.saved.kappa .* (sigma .+ result.saved.sigmap)
    r = result.saved.r[end]
    Sbar = sigma * sqrt(gamma / kappas)
    sbar = log(Sbar)
    smax = sbar + 0.5 * (1 - Sbar^2)
    function interpolate(y, x)
        i = clamp(searchsortedlast(ss, x), 1, length(ss) - 1)
        w = (x - ss[i]) / (ss[i + 1] - ss[i])
        (1 - w) * y[i] + w * y[i + 1]
    end
    dt, rng, s = 1 / 12, Xoshiro(seed), sbar
    gross, growth, ratio, conditional = zeros(months), zeros(months), zeros(months), zeros(months)
    for t in 1:months
        z = randn(rng)
        lambda = s >= smax ? 0.0 : 1 / Sbar * sqrt(1 - 2 * (s - sbar)) - 1
        snew = min(s - kappas * (s - sbar) * dt + lambda * sigma * sqrt(dt) * z, smax)
        growth[t] = mu * dt + sigma * sqrt(dt) * z
        gross[t] = (interpolate(p, snew) + dt) / interpolate(p, s) * exp(growth[t])
        ratio[t] = interpolate(p, snew)
        conditional[t] = interpolate(premium, s)
        s = snew
    end
    years = div(months, 12)
    within(y) = (12 * (y - 1) + 1):(12 * y)
    logc = cumsum(growth)
    logannual = [log(sum(exp.(logc[within(y)] .- logc[12 * y])) * dt) + logc[12 * y] for y in 1:years]
    logpd = [log(ratio[12 * y]) + logc[12 * y] - logannual[y] for y in 1:years]
    annualreturn = [prod(gross[within(y)]) for y in 1:years]
    logexcess = log.(annualreturn) .- r
    simpleexcess = annualreturn .- exp(r)
    (; Edc = 100 * mean(diff(logannual)), sdc = 100 * std(diff(logannual)), rf = 100 * r,
       sharpelog = mean(logexcess) / std(logexcess), sharpesimple = mean(simpleexcess) / std(simpleexcess),
       premium = 100 * mean(logexcess), volatility = 100 * std(logexcess),
       pd = exp(mean(logpd)), sdpd = std(logpd),
       realised = 1200 * mean(gross .- exp(r * dt)), conditional = 100 * mean(conditional))
end

a = table2_statistics(m, result, ss)
println("Mean conditional premium: ", round(a.conditional, digits = 2), "% a year")
println("Mean realised simple excess return: ", round(a.realised, digits = 2), "% a year")

The second pair of lines is a check. The mean of the premium formula along the path and the mean excess return realised along the same path are independent routes to one number, and they agree.

### Two discount rates

The example's `rho = 0.116` gives a risk-free rate of 1.58%, while the paper's is .94%. Campbell and Cochrane's Table 1 prints `delta = .89`, close to `exp(-0.116)`, but .94% needs `delta` of about .896, and Table 1 of [Wachter (2005)](https://doi.org/10.1016/j.frl.2005.08.001) prints .90. So the table has a second column, with `rho` set for a risk-free rate of .94%.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

rho094 = 0.0094 - m.gamma * m.mu + m.gamma * m.kappas / 2
m094 = CampbellCochraneModel(rho = rho094)
result094 = pdesolve(m094, stategrid, guess; verbose = false)
b = table2_statistics(m094, result094, ss)
println("Converged: ", result094.converged, "; rho = ", round(rho094, digits = 4), "; delta = ", round(exp(-rho094), digits = 3))

# Published values: Campbell and Cochrane (1999), Table 2, Consumption Claim; Wachter (2005), Table 2, fixed-point method, Grids 1 and 3
rows = [("E(dc)", :Edc, 2, "1.89", "", ""),
        ("sd(dc)", :sdc, 2, "1.22", "", ""),
        ("E(r^f)", :rf, 2, ".094", "0.94", "0.94"),
        ("E(r-r^f)/sd(r-r^f)", :sharpelog, 2, ".43", "0.44", "0.47"),
        ("E(R-R^f)/sd(R-R^f)", :sharpesimple, 2, ".50", "", ""),
        ("E(r-r^f)", :premium, 2, "6.64", "6.59", "3.89"),
        ("sd(r-r^f)", :volatility, 2, "15.2", "15.05", "8.23"),
        ("exp[E(p-d)]", :pd, 1, "18.3", "18.62", "34.66"),
        ("sd(p-d)", :sdpd, 2, ".27", "0.27", "0.13")]
fixed(x, digits) = digits == 1 ? @sprintf("%.1f", x) : @sprintf("%.2f", x)
println(rpad("", 20), rpad("rho .116", 10), rpad("r^f .94", 10), rpad("C and C", 10), rpad("Grid 1", 10), "Grid 3")
for (label, key, digits, cc, grid1, grid3) in rows
    println(rpad(label, 20), rpad(fixed(getfield(a, key), digits), 10), rpad(fixed(getfield(b, key), digits), 10), rpad(cc, 10), rpad(grid1, 10), grid3)
end

**C and C** is the Consumption Claim column of Table 2 of Campbell and Cochrane (p. 225). **Grid 1** and **Grid 3** are from Table 2 of Wachter, with the fixed-point method; these were read in the [NBER Working Paper 11559](https://www.nber.org/papers/w11559) version, and the tables of the published article have not been compared.

Table 2 of Campbell and Cochrane prints .094 for `E(r^f)`. Their Table 1 gives .94, as does Wachter's Table 2, so .094 looks like a misprint.

### Why the columns differ

The Sharpe ratios stay close to the values the paper chose its parameters to match: .47 here against .43 for log returns, .50 against .50 for simple returns. The levels do not. The premium and its volatility are 3.85 and 8.19 here, against 6.64 and 15.2 in Campbell and Cochrane, and the postwar sample of their Table 2 has 6.69 and 15.7.

The difference is not monthly against continuous time. [Wachter (2005)](https://doi.org/10.1016/j.frl.2005.08.001) solved the monthly model on several grids. Grid 1, the 17 points Campbell and Cochrane used, gives 6.59 and 15.05. Grid 3, with 101 points equally spaced in `S` and 900 points logarithmically spaced down to `s = -300`, gives 3.89 and 8.23. The package's grid is of that second kind, with 100 points equally spaced in `S` and 900 in `s` down to -300, and the values here are next to Grid 3.

So at these parameters, solved on a fine grid, the premium of the consumption claim is well below the 6.69 of the postwar sample.

## Source and Licence

The code under **From EconPDEs.jl** is adapted from `examples/asset_pricing/campbell_cochrane.jl` of [EconPDEs.jl](https://github.com/matthieugomez/EconPDEs.jl), version 3.1.0, by Matthieu Gomez. Changes made in Pollis: Greek letters written in Latin; the model function also returns `kappa`, `sigmap` and `r`.

Added in Pollis, not in the package: the plots of the equity premium and the Sharpe ratio, the simulation of Table 2 with its check, the second discount rate, and the comparison with Campbell and Cochrane and with Wachter.

The notice below covers the package's code only, not the parts added in Pollis.

The EconPDEs.jl package is licensed under the MIT "Expat" License:

> Copyright (c) 2015: Matthieu Gomez.
>
> Permission is hereby granted, free of charge, to any person obtaining
> a copy of this software and associated documentation files (the
> "Software"), to deal in the Software without restriction, including
> without limitation the rights to use, copy, modify, merge, publish,
> distribute, sublicense, and/or sell copies of the Software, and to
> permit persons to whom the Software is furnished to do so, subject to
> the following conditions:
>
> The above copyright notice and this permission notice shall be
> included in all copies or substantial portions of the Software.
>
> THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
> EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
> MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
> IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
> CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
> TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
> SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## References

Campbell, J. Y. and Cochrane, J. H. (1999). By force of habit: a consumption-based explanation of aggregate stock market behavior. *Journal of Political Economy* 107(2), 205–251. [doi:10.1086/250059](https://doi.org/10.1086/250059)

Wachter, J. A. (2005). Solving models with external habit. *Finance Research Letters* 2(4), 210–226. [doi:10.1016/j.frl.2005.08.001](https://doi.org/10.1016/j.frl.2005.08.001)